# Barebones SpatialData Image Write Test (`bioio-ome-tiff`, 10k crop)

This notebook is the BioIO-reader counterpart to the manual `Image2DModel` and `sopa.io.ome_tif` image-only tests.

It does only five things:

1. inspect the merged OME-TIFF metadata and TIFF tiling
2. load the image with `BioImage`
3. inspect the Dask chunking exposed by BioIO
4. crop to a 10k x 10k region, construct `SpatialData(images={...})`, and write it
5. reopen the written store and inspect the result

The key question is whether BioIO exposes a better source chunk layout than `dask_image.imread(...)` for this OME-TIFF.


In [1]:
from __future__ import annotations

import os
import json
import shutil
from pathlib import Path

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

import bioio
import bioio_ome_tiff
import spatialdata
import tifffile
from bioio import BioImage
from bioio_ome_tiff import Reader
from spatialdata import SpatialData, read_zarr
from spatialdata.models import Image2DModel

print("bioio", bioio.__version__)
print("bioio_ome_tiff", bioio_ome_tiff.__version__)
print("spatialdata", spatialdata.__version__)
print("tifffile", tifffile.__version__)


/home/ratnayn/miniconda3/envs/sdata_bioio/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


bioio 3.3.0
bioio_ome_tiff 1.4.0
spatialdata 0.7.0
tifffile 2026.3.3


/home/ratnayn/miniconda3/envs/sdata_bioio/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
OUTPUTS_DIR = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs")
FULL_MERGE_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329_full_merge.ome.tif")
CHANNEL_MAP_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json")
WRITE_PATH = OUTPUTS_DIR / "spatialdata_image_write_only_bioio_ome_tiff_SLIDE-0329_crop_2000.sdata.zarr"

CROP_X = 0
CROP_Y = 0
CROP_WIDTH = 2000
CROP_HEIGHT = 2000
CHUNK_SHAPE = (1, 512, 512)
SCALE_FACTORS = [2, 2, 2, 2]
BIOIO_CHUNK_DIMS = ["C"]

print(FULL_MERGE_PATH)
print(CHANNEL_MAP_PATH)
print(WRITE_PATH)
print({
    "crop": (CROP_X, CROP_Y, CROP_WIDTH, CROP_HEIGHT),
    "chunk_shape": CHUNK_SHAPE,
    "scale_factors": SCALE_FACTORS,
    "bioio_chunk_dims": BIOIO_CHUNK_DIMS,
})


/mnt/c/Analysis/Data_prototype/SLIDE-0329_full_merge.ome.tif
/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json
/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/spatialdata_image_write_only_bioio_ome_tiff_SLIDE-0329_crop_2000.sdata.zarr
{'crop': (0, 0, 2000, 2000), 'chunk_shape': (1, 512, 512), 'scale_factors': [2, 2, 2, 2], 'bioio_chunk_dims': ['C']}


In [3]:
channel_map = json.loads(CHANNEL_MAP_PATH.read_text())
channel_aliases = [entry["alias"] for entry in channel_map]

with tifffile.TiffFile(FULL_MERGE_PATH) as tif:
    series = tif.series[0]
    level0 = series.levels[0]
    page0 = level0.pages[0]
    tile_info = {
        "is_tiled": page0.is_tiled,
        "tilewidth": page0.tilewidth,
        "tilelength": page0.tilelength,
        "compression": str(page0.compression),
    }
    print("series axes:", series.axes)
    print("series shape:", series.shape)
    print("levels:", len(series.levels))
    print("level0 shape:", level0.shape)
    print("tile_info:", tile_info)

print("channel count:", len(channel_aliases))
print("first aliases:", channel_aliases[:6])


series axes: CYX
series shape: (25, 62617, 66406)
levels: 8
level0 shape: (25, 62617, 66406)
tile_info: {'is_tiled': True, 'tilewidth': 256, 'tilelength': 256, 'compression': '8'}
channel count: 25
first aliases: ['R1_DAPI_AF', 'R1_BIT2_RS0584_CY3B', 'R1_BIT3_RS0015_CY5', 'R1_BIT4_RS0083_750', 'R1_DAPI', 'R1_BIT1_RS0996_488']


In [4]:
img = BioImage(FULL_MERGE_PATH, reader=Reader, chunk_dims=BIOIO_CHUNK_DIMS)
print("reader:", type(img.reader).__name__)
print("shape:", img.shape)
print("dims:", img.dims)
print("scenes:", img.scenes)

lazy_image_array = img.get_image_dask_data("CYX")
print("lazy_image_array:", lazy_image_array)
print("lazy_image_array.shape:", lazy_image_array.shape)
print("lazy_image_array.chunks:", lazy_image_array.chunks)
print("lazy_image_array.chunksize:", lazy_image_array.chunksize)
print("lazy_image_array.dtype:", lazy_image_array.dtype)


reader: Reader
shape: (1, 25, 1, 62617, 66406)
dims: <Dimensions [T: 1, C: 25, Z: 1, Y: 62617, X: 66406]>
scenes: ('Image:0',)
lazy_image_array: dask.array<getitem, shape=(25, 62617, 66406), dtype=uint16, chunksize=(25, 62617, 66406), chunktype=numpy.ndarray>
lazy_image_array.shape: (25, 62617, 66406)
lazy_image_array.chunks: ((25,), (62617,), (66406,))
lazy_image_array.chunksize: (25, 62617, 66406)
lazy_image_array.dtype: uint16


In [5]:
cropped_image_array = lazy_image_array[:, CROP_Y:CROP_Y + CROP_HEIGHT, CROP_X:CROP_X + CROP_WIDTH]
cropped_image_array = cropped_image_array.rechunk(CHUNK_SHAPE)

print("cropped_image_array:", cropped_image_array)
print("cropped_image_array.shape:", cropped_image_array.shape)
print("cropped_image_array.chunks:", cropped_image_array.chunks)
print("cropped_image_array.chunksize:", cropped_image_array.chunksize)

full_image = Image2DModel.parse(
    cropped_image_array,
    dims=("c", "y", "x"),
    c_coords=channel_aliases,
    chunks=CHUNK_SHAPE,
    scale_factors=SCALE_FACTORS,
)

sdata = SpatialData(images={"full_image": full_image})
sdata


cropped_image_array: dask.array<rechunk-merge, shape=(25, 2000, 2000), dtype=uint16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>
cropped_image_array.shape: (25, 2000, 2000)
cropped_image_array.chunks: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (512, 512, 512, 464), (512, 512, 512, 464))
cropped_image_array.chunksize: (1, 512, 512)


SpatialData object
└── Images
      └── 'full_image': DataTree[cyx] (25, 2000, 2000), (25, 1000, 1000), (25, 500, 500), (25, 250, 250), (25, 125, 125)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images)

In [7]:
scale0 = sdata.images["full_image"]["scale0"].image
print("scale0 shape:", scale0.shape)
print("scale0 chunks:", scale0.chunks)
print("scale0 data chunksize:", scale0.data.chunksize)
print("scale0 channel coords:", scale0.coords["c"].values[:6].tolist())



scale0 shape: (25, 2000, 2000)
scale0 chunks: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (512, 512, 512, 464), (512, 512, 512, 464))
scale0 data chunksize: (1, 512, 512)
scale0 channel coords: ['R1_DAPI_AF', 'R1_BIT2_RS0584_CY3B', 'R1_BIT3_RS0015_CY5', 'R1_BIT4_RS0083_750', 'R1_DAPI', 'R1_BIT1_RS0996_488']


In [8]:
if WRITE_PATH.exists():
    shutil.rmtree(WRITE_PATH)

sdata.write(WRITE_PATH, overwrite=True)
print("Wrote:", WRITE_PATH)
print("Exists:", WRITE_PATH.exists())


TypeError: Expected an iterable of integers. Got ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (512, 512, 512, 464), (512, 512, 512, 464)) instead.

In [ ]:
reloaded = read_zarr(WRITE_PATH)
print(reloaded)
print("images", list(reloaded.images.keys()))
print("labels", list(reloaded.labels.keys()))
print("tables", list(reloaded.tables.keys()))
